# Detecção de Spam e Phishing — Classificação de Emails

Pipeline end-to-end de **processamento de linguagem natural (NLP)** e classificação binária, construído de raiz no Databricks, célula a célula, com validação do utilizador em cada etapa.

**Dataset:** público, do Kaggle — `isuranga/spammalicious-detection` ("Email Classification Data"). Contém **100.000 emails reais de 2024** com **30 colunas**, abrangendo o texto completo do email, assunto, remetente, domínio, número de links, anexos, tokens de rastreio (tracking), score do SpamAssassin e a etiqueta binária alvo (`spam` / `legítimo`). O corpus é **multilingue** (inglês, alemão, italiano, entre outras línguas), o que enriquece o desafio de engenharia de features.

**Objetivo:** treinar modelos de classificação (Naive Bayes e Random Forest) capazes de identificar automaticamente emails maliciosos, combinando features textuais (TF-IDF sobre o texto limpo) com features comportamentais (links, anexos, domínio do remetente, score SpamAssassin), e responder a sete perguntas de negócio sobre os padrões de spam.

**Métrica de sucesso (definida desde o início):**

| Métrica | Objetivo mínimo | Justificação |
| --- | --- | --- |
| F1-score (classe spam) | ≥ 0,90 | Métrica principal — equilibra precisão e revocação num problema onde os dois tipos de erro custam caro |
| Acurácia | ≥ 0,90 | Visão global, fiável num dataset relativamente equilibrado |
| Recall (classe spam) | ≥ 0,85 | O pior cenário em segurança é deixar passar um phishing (falso negativo) |

**Metodologia:** divisão treino/teste desde o início (evita *data leakage*), limpeza de texto com NLTK, extração de features com TF-IDF, treino e comparação de dois classificadores, avaliação com F1-score, acurácia e matriz de confusão, seguida de sete células de perguntas de negócio respondidas com os dados.

**Resultado final:** nos 20.000 emails de teste — nunca vistos pelos modelos — Naive Bayes e Random Forest acertaram em todos: **F1 = acurácia = recall = precisão = 1,000**, zero erros na matriz de confusão. As três metas acima foram cumpridas. Mas um resultado perfeito pede sempre uma verificação de honestidade: essa investigação, e o que ela revelou sobre o dataset, está na **Conclusão**.

In [0]:
# ==============================================================================
# STEP 0: INSTALAÇÃO DE DEPENDÊNCIAS NO AMBIENTE
# ==============================================================================
# Objectivo: instalar as bibliotecas necessárias ao projeto no ambiente serverless.
#            Fica no topo do notebook para ser sempre a 1.ª célula executada —
#            replicável em qualquer sessão nova (Git/Kaggle).
#
# Input:  nenhum (bibliotecas do PyPI)
# Output: ambiente com kagglehub (download do dataset) e nltk (limpeza de texto)
#
# Operações:
#   - Instalar kagglehub — download de datasets públicos do Kaggle SEM chave de API
#   - Instalar nltk — stopwords, tokenização e lematização na fase de limpeza
#   - Reiniciar o interpretador Python para as bibliotecas ficarem visíveis
#
# Nota: pandas, scikit-learn e matplotlib já vêm pré-instalados no Databricks.
# Os recursos do NLTK (stopwords, etc.) são descarregados na STEP da limpeza.

%pip install -q kagglehub nltk

print('Bibliotecas instaladas — prontas a importar.')

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ==============================================================================
# STEP 1: DOWNLOAD DO DATASET DO KAGGLE E CARREGAMENTO EM DATAFRAME
# ==============================================================================
# Objectivo: obter o dataset isuranga/spammalicious-detection ("Email
#            Classification Data": 100.000 emails reais de 2024, 30 colunas,
#            multilingue) e carregá-lo em pandas, com verificação de estrutura.
#
# Input:  dataset público do Kaggle — anexado ao notebook (Kaggle "Add Input")
#         ou descarregado via kagglehub (Databricks/Jupyter, sem chave de API)
# Output: df (DataFrame pandas) + caminho do CSV utilizado
#
# Operações:
#   - Procurar o CSV nos datasets anexados (/kaggle/input, no Kaggle)
#   - Se não existir, descarregar o dataset com kagglehub (cache local)
#   - Carregar em pandas e validar: nº de linhas, nº de colunas, tipos (dtypes)

import os
import glob
import kagglehub
import pandas as pd

# 1) Dataset anexado ao notebook? (Kaggle: "Add Input" — sem download)
anexos = glob.glob("/kaggle/input/**/email_dataset_100k.csv", recursive=True)
if anexos:
    csv_path = anexos[0]
    print(f"Dataset anexado ao notebook: {csv_path}")
# 2) Caso contrário, descarrega o dataset público (sem credenciais)
else:
    if os.path.exists("/kaggle/input"):
        print("Aviso: no Kaggle, anexa o dataset ao notebook (Add Input) — o "
              "download via kagglehub não é suportado em sessões não interativas.")
    dataset_path = kagglehub.dataset_download("isuranga/spammalicious-detection")
    csv_path = os.path.join(dataset_path, "email_dataset_100k.csv")
    print(f"Dataset descarregado para: {dataset_path}")

# Carrega em DataFrame pandas
df = pd.read_csv(csv_path)

# Verificação de estrutura e esquema (100.000 linhas × 30 colunas esperado)
print(f"\nDimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
print("\nEsquema (coluna, tipo, valores não nulos):")
print(df.info())

100%|██████████| 12.0M/12.0M [00:00<00:00, 32.0MB/s]

Extracting files...


Dataset em: /home/spark-d6e8af4c-925f-47ef-a180-49/.cache/kagglehub/datasets/isuranga/spammalicious-detection/versions/1
Ficheiros: ['email_dataset_100k.csv']

Dimensões: 100000 linhas × 30 colunas

Esquema (coluna, tipo, valores não nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   raw_text                 100000 non-null  object 
 1   subject                  100000 non-null  object 
 2   body_plain               100000 non-null  object 
 3   body_html                42403 non-null   object 
 4   from_address             100000 non-null  object 
 5   from_domain              100000 non-null  object 
 6   reply_to                 29918 non-null   object 
 7   to_addresses             100000 non-null  object 
 8   cc_addresses             19902 non-null   object 
 9   date                     100000 non-nu

In [0]:
# ==============================================================================
# STEP 2: EXPLORAÇÃO DE METADADOS E ANÁLISE EXPLORATÓRIA (EDA)
# ==============================================================================
# Objectivo: caracterizar a variável alvo (label), as línguas e as features
#            comportamentais, para perceber o equilíbrio das classes e o poder
#            discriminativo de cada feature antes da modelação.
#
# Input:  df (100.000 emails × 30 colunas)
# Output: tabelas de distribuição da label, línguas, autenticação e
#         estatísticas das features numéricas por classe
#
# Operações:
#   - Distribuição da label (spam vs legítimo) em contagem e percentagem
#   - Distribuição das línguas e taxa de spam por língua
#   - Resultados de autenticação (SPF/DKIM/DMARC) por classe
#   - Estatísticas descritivas das features numéricas, separadas por classe

# --- Distribuição da variável alvo -------------------------------------------
# 1 = spam, 0 = legítimo (a confirmar com as contagens)
print("" + "=" * 60)
print("DISTRIBUIÇÃO DA LABEL (alvo)")
print("=" * 60)
print(df["label"].value_counts())
print((df["label"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

# --- Línguas -------------------------------------------------------------------
print("\n" + "=" * 60)
print("LÍNGUAS (contagem e taxa de spam por língua)")
print("=" * 60)
taxa_linguas = df.groupby("language")["label"].agg(total="count", taxa_spam="mean")
taxa_linguas["taxa_spam_pct"] = (taxa_linguas["taxa_spam"] * 100).round(2)
taxa_linguas["total_pct"] = (taxa_linguas["total"] / len(df) * 100).round(2)
display(taxa_linguas.drop(columns="taxa_spam").sort_values("total", ascending=False))

# --- Autenticação de email por classe -----------------------------------------
print("\n" + "=" * 60)
print("AUTENTICAÇÃO (SPF / DKIM / DMARC) vs CLASSE")
print("=" * 60)
for col in ["spf_result", "dkim_result", "dmarc_result"]:
    tabela = pd.crosstab(df[col], df["label"], normalize="index") * 100
    tabela.columns = ["% legítimo", "% spam"]
    print(f"\n{col}:")
    print(tabela.round(2))

# --- Features comportamentais numéricas por classe ----------------------------
print("\n" + "=" * 60)
print("FEATURES COMPORTAMENTAIS — MÉDIA POR CLASSE")
print("=" * 60)
feats_num = ["num_urls", "num_emails_in_body", "num_phone_numbers",
             "num_received_headers", "x_spam_score", "hour_of_day"]
medias = df.groupby("label")[feats_num].mean().round(2)
medias.index = ["legítimo", "spam"]
display(medias.T.rename(columns={"legítimo": "média em legítimos", "spam": "média em spam"}))

# --- Booleanas por classe -----------------------------------------------------
print("\n" + "=" * 60)
print("FEATURES BOOLEANAS — % VERDADEIRO POR CLASSE")
print("=" * 60)
feats_bool = ["has_attachments", "has_html", "contains_tracking_token"]
bool_pct = df.groupby("label")[feats_bool].mean() * 100
bool_pct.index = ["legítimo", "spam"]
display(bool_pct.T.round(2))

DISTRIBUIÇÃO DA LABEL (alvo)
label
0    50000
1    50000
Name: count, dtype: int64
label
0    50.0 %
1    50.0 %
Name: proportion, dtype: object

LÍNGUAS (contagem e taxa de spam por língua)


total,taxa_spam_pct,total_pct
60032,49.93,60.03
10041,50.44,10.04
10012,49.29,10.01
9981,49.69,9.98
9934,51.02,9.93



AUTENTICAÇÃO (SPF / DKIM / DMARC) vs CLASSE

spf_result:
            % legítimo  % spam
spf_result                    
fail             16.48   83.52
neutral          24.53   75.47
pass             80.25   19.75

dkim_result:
             % legítimo  % spam
dkim_result                    
fail              16.42   83.58
neutral           24.80   75.20
pass              79.95   20.05

dmarc_result:
              % legítimo  % spam
dmarc_result                    
fail               20.59   79.41
none               33.19   66.81
pass               77.65   22.35

FEATURES COMPORTAMENTAIS — MÉDIA POR CLASSE


média em legítimos,média em spam
0.0,1.6
0.0,0.08
0.18,1.08
5.01,4.0
0.0,8.0
11.51,11.47



FEATURES BOOLEANAS — % VERDADEIRO POR CLASSE


legítimo,spam
15.13,25.15
24.87,59.94
4.99,60.36


In [0]:
# ==============================================================================
# STEP 3: DIVISÃO TREINO/TESTE ESTRATIFICADA (80/20) — ANTES DE TUDO
# ==============================================================================
# Objectivo: separar 80% treino / 20% teste com estratificação pela label, ANTES
#            da limpeza e da extração de features. Tudo o que "aprende" com os
#            dados (TF-IDF, stopwords, modelos) é ajustado SÓ no treino — o
#            teste simula emails nunca vistos e evita data leakage.
#
# Input:  df (100.000 emails × 30 colunas)
# Output: df_treino (80.000) e df_teste (20.000), com a mesma proporção 50/50
#
# Operações:
#   - train_test_split estratificado pela label, com random_state fixo
#   - Verificação: contagens e proporções de classes em cada conjunto

from sklearn.model_selection import train_test_split

# Divisão estratificada: mantém 50/50 de spam/legítimo em treino e teste
df_treino, df_teste = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
df_treino = df_treino.reset_index(drop=True)
df_teste = df_teste.reset_index(drop=True)

# Verificação: dimensões e proporção de classes em cada conjunto
print(f"Treino: {df_treino.shape[0]} emails | Teste: {df_teste.shape[0]} emails")
print("\nProporção de classes (treino):")
print((df_treino["label"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")
print("\nProporção de classes (teste):")
print((df_teste["label"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

Treino: 80000 emails | Teste: 20000 emails

Proporção de classes (treino):
label
0    50.0 %
1    50.0 %
Name: proportion, dtype: object

Proporção de classes (teste):
label
1    50.0 %
0    50.0 %
Name: proportion, dtype: object


In [0]:
# ==============================================================================
# STEP 2B: VISUALIZAÇÃO DA ESTRUTURA DO DATASET (77 LINHAS × 30 COLUNAS)
# ==============================================================================
# Objectivo: mostrar as primeiras 77 linhas do dataset com TODAS as colunas
#            visíveis, para inspecionar a estrutura real dos dados — conteúdo
#            das colunas de texto, remetentes, autenticação, features
#            comportamentais e a própria label.
#
# Input:  df (100.000 emails × 30 colunas)
# Output: tabela interativa com 77 emails e as 30 colunas (usar a barra de
#         deslocamento horizontal para ver todas; clicar numa célula expande)
#
# Operações:
#   - Garantir que o pandas mostra todas as colunas (sem truncar)
#   - display() das primeiras 77 linhas do dataset

# Mostrar todas as colunas sem truncar
pd.set_option("display.max_columns", None)

# Primeiras 77 linhas — estrutura completa do dataset
display(df.head(77))

raw_text,subject,body_plain,body_html,from_address,from_domain,reply_to,to_addresses,cc_addresses,date,hour_of_day,message_id,in_reply_to,num_received_headers,received_origin_ip,spf_result,dkim_result,dmarc_result,has_attachments,attachment_types,has_html,num_urls,num_emails_in_body,num_phone_numbers,contains_tracking_token,x_spam_score,user_agent,list_unsubscribe,language,label
"Subject: Release notes v2.6.4 Dear Vik, We received invoice 765713. It will be processed by Accounts Payable. Best, Wendy",Release notes v2.6.4,"Dear Vik, We received invoice 765713. It will be processed by Accounts Payable. Best, Wendy",null,leo.wilson@adatum.net,adatum.net,null,kathy_taylor@notifications.net,null,2024-09-13T16:25:32,16,,,7,155.60.143.1,fail,fail,none,true,report_314379.pdf;agenda_095693.ics;confirm_084392.pdf;confirm_882079.ics,false,0,0,1,false,0.0,Microsoft Outlook,null,en,0
"Subject: WINNER! Claim your reward Dear user, Your parcel has been flagged. To restore access, click the link below: https://update.info/billing/94332769 Failure to act may result in closure. Support",WINNER! Claim your reward,"Dear user, Your parcel has been flagged. To restore access, click the link below: https://update.info/billing/94332769 Failure to act may result in closure. Support",null,riya_miller313@proton.me,proton.me,null,olivia_wilson841@accounts-mail.com,null,2024-07-10T01:15:32,1,,null,5,84.17.122.15,pass,fail,none,false,null,false,1,0,1,false,4.82,Mutt,null,en,1
"Subject: Follow-up: onboarding tasks Hi Zack, Here are the notes from our meeting. Please review the attached document and add comments. Thanks, Frank",Follow-up: onboarding tasks,"Hi Zack, Here are the notes from our meeting. Please review the attached document and add comments. Thanks, Frank",null,wendytaylor@cohosu.org,cohosu.org,null,vikwilliams@contoso.com,null,2024-04-23T16:20:32,16,,null,4,201.192.196.145,pass,pass,pass,false,null,false,0,0,0,false,0.0,Apple Mail,null,de,0
"Subject: Limited-time offer: $119 cloud storage Attention Bob, We detected unusual activity on your cloud storage. Verify immediately: https://notify-vkay.com/auth/03088164-8842 If you ignore this message, your data may be at risk.",Limited-time offer: $119 cloud storage,"Attention Bob, We detected unusual activity on your cloud storage. Verify immediately: https://notify-vkay.com/auth/03088164-8842 If you ignore this message, your data may be at risk.","Limited-time offer: $119 cloud storageAttention Bob,We detected unusual activity on your cloud storage. Verify immediately:https://notify-vkay.com/auth/03088164-8842If you ignore this message, your data may be at risk.",vik.brown@outlook.com,outlook.com,carol.johnson@icloud.com,carol.rodriguez@devops.local,null,2024-06-07T19:15:32,19,,null,6,9.214.83.251,neutral,fail,none,false,null,true,2,0,1,false,9.15,Mutt,null,it,1
"Subject: Weekly update - Legal Team, Notes from Legal daily standup attached. Please update your tasks in the tracker. Thanks, Eve",Weekly update - Legal,"Team, Notes from Legal daily standup attached. Please update your tasks in the tracker. Thanks, Eve","Team,Notes from Legal daily standup attached. Please update your tasks in the tracker.Thanks,Eve",grace_taylor@globalbank.com,globalbank.com,null,leo_wilson@notifications.net,null,2024-07-05T11:33:32,11,,null,5,187.16.2.98,pass,pass,pass,false,null,true,0,0,0,false,0.0,Thunderbird,null,it,0
Subject: Your parcel is waiting (ID 381863) IMPORTANT: Submit your details to claim reward: https://support.co/track/03148545-kkwcu This is a limited offer. Ref: ref-QMW5680416,Your parcel is waiting (ID 381863),IMPORTANT: Submit your details to claim reward: https://support.co/track/03148545-kkwcu This is a limited offer. Ref: ref-QMW5680416,null,frankwilson76@bizsupport.co,bizsupport.co,null,sara_johnson@icloud.com,null,2024-04-03T16:03:32,16,,null,6,33.7.100.153,neutral,neutral,pass,false,null,false,1,0,1,true,10.99,Outlook-Android,https://notify.app/wallet/01093231-sqtas,en,1
"Subject: Lunch tomorro

In [0]:
# ==============================================================================
# STEP 4: LIMPEZA DE TEXTO COM NLTK (MULTILINGUE)
# ==============================================================================
# Objectivo: normalizar o texto (assunto + corpo) para a extração de features:
#            o TF-IDF só funciona bem com texto limpo — sem HTML, URLs, emails,
#            pontuação, maiúsculas nem stopwords nas 5 línguas do dataset.
#
# Input:  df_treino e df_teste (colunas "subject" e "body_plain")
# Output: coluna "texto_limpo" em ambos os conjuntos (treino e teste)
#
# Operações:
#   - Descarregar stopwords do NLTK nas 5 línguas (en, fr, it, es, de)
#   - Função limpar_texto(): minúsculas, remover HTML, URLs, emails e pontuação
#   - Aplicar a treino e teste separadamente; validar com exemplos antes/depois

import re
import nltk
from nltk.corpus import stopwords

# Stopwords nas 5 línguas do dataset (en, fr, it, es, de)
nltk.download("stopwords", quiet=True)
stop_words = set()
for lingua in ["english", "french", "italian", "spanish", "german"]:
    stop_words |= set(stopwords.words(lingua))
print(f"Stopwords carregadas: {len(stop_words)} (5 línguas)")


def limpar_texto(texto):
    """Normaliza um email: minúsculas, sem HTML/URLs/emails/pontuação,
    sem stopwords. Devolve uma string de tokens separados por espaço."""
    texto = texto.lower()                          # minúsculas
    texto = re.sub(r"<[^>]+>", " ", texto)         # tags HTML
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)  # URLs
    texto = re.sub(r"\S+@\S+", " ", texto)        # endereços de email
    texto = re.sub(r"[^a-zà-öø-ÿ\s]", " ", texto)  # só letras acentuadas
    tokens = [t for t in texto.split()
              if t not in stop_words and len(t) > 1]  # remove stopwords
    return " ".join(tokens)


# Texto a limpar = assunto + corpo em texto simples (body_plain)
for conjunto in (df_treino, df_teste):
    conjunto["texto_limpo"] = (
        conjunto["subject"].fillna("") + " " + conjunto["body_plain"].fillna("")
    ).apply(limpar_texto)

# Validação: exemplo antes/depois (1 spam e 1 legítimo) e textos vazios
print("=" * 60)
print("EXEMPLO — SPAM (label 1)")
linha = df_treino[df_treino["label"] == 1].iloc[0]
print("Antes :", (linha["subject"] + " " + linha["body_plain"])[:150])
print("Depois:", linha["texto_limpo"][:150])
print("\nEXEMPLO — LEGÍTIMO (label 0)")
linha = df_treino[df_treino["label"] == 0].iloc[0]
print("Antes :", (linha["subject"] + " " + linha["body_plain"])[:150])
print("Depois:", linha["texto_limpo"][:150])

vazios_treino = (df_treino["texto_limpo"].str.len() == 0).sum()
vazios_teste = (df_teste["texto_limpo"].str.len() == 0).sum()
print(f"\nTextos vazios após limpeza: treino={vazios_treino}, teste={vazios_teste}")
print(f"Comprimento médio (treino): {df_treino['texto_limpo'].str.split().str.len().mean():.1f} palavras")

Stopwords carregadas: 1108 (5 línguas)
EXEMPLO — SPAM (label 1)
Antes : Exclusive deal on account today Dear user,

Your account has been flagged. To restore access, click the link below:
https://support.app/verify/4463285
Depois: exclusive deal account today dear user account flagged restore access click link failure act may result closure support ref id pwl

EXEMPLO — LEGÍTIMO (label 0)
Antes : Invoice 140881 received Team,

Notes from Operations daily standup attached. Please update your tasks in the tracker.

Thanks,
Zack
Depois: invoice received team notes operations daily standup attached please update tasks tracker thanks zack

Textos vazios após limpeza: treino=0, teste=0
Comprimento médio (treino): 13.9 palavras


In [0]:
# ==============================================================================
# STEP 5: EXTRAÇÃO DE FEATURES — TF-IDF (TEXTO) + COMPORTAMENTAIS
# ==============================================================================
# Objectivo: converter os emails em matrizes numéricas para os modelos:
#            (a) TF-IDF sobre o texto limpo — vocabulário aprendido SÓ no treino
#            (b) features comportamentais — links, anexos, score SpamAssassin,
#                autenticação SPF/DKIM/DMARC e língua (one-hot)
#
# Input:  df_treino e df_teste (colunas "texto_limpo" + features brutas)
# Output: X_treino, X_teste (matrizes esparsas combinadas) e y_treino, y_teste
#
# Operações:
#   - TfidfVectorizer: fit no treino, transform no teste (zero leakage)
#   - Matriz comportamental: numéricas + booleanas + one-hot (alinhada pelo treino)
#   - Empilhar tudo (texto + comportamento) numa matriz esparsa única
#   - Validar dimensões e mostrar os termos TF-IDF mais fortes no spam

import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

# --- (a) TF-IDF do texto — ajustado apenas no treino ---------------------------
vectorizador = TfidfVectorizer(max_features=5000, min_df=2, ngram_range=(1, 2))
X_texto_treino = vectorizador.fit_transform(df_treino["texto_limpo"])
X_texto_teste = vectorizador.transform(df_teste["texto_limpo"])  # só transform
print(f"Vocabulário TF-IDF: {len(vectorizador.vocabulary_)} termos (unigramas + bigramas)")

# --- (b) Matriz comportamental --------------------------------------------------
cols_num = ["num_urls", "num_emails_in_body", "num_phone_numbers",
            "num_received_headers", "x_spam_score", "hour_of_day"]
cols_bool = ["has_attachments", "has_html", "contains_tracking_token"]
cols_cat = ["spf_result", "dkim_result", "dmarc_result", "language"]


def matriz_comportamental(d, colunas_referencia=None):
    """Junta numéricas + booleanas + one-hot das categóricas.
    Se der colunas de referência (do treino), alinha por elas (fill 0)."""
    num = d[cols_num].astype(float)
    boo = d[cols_bool].astype(int)
    cat = pd.get_dummies(d[cols_cat], columns=cols_cat).astype(float)
    comp = pd.concat([num, boo, cat], axis=1)
    if colunas_referencia is not None:
        comp = comp.reindex(columns=colunas_referencia, fill_value=0)
    return comp


# Treino define as colunas de referência; o teste alinha-se a elas
comp_treino = matriz_comportamental(df_treino)
comp_teste = matriz_comportamental(df_teste, colunas_referencia=comp_treino.columns)
print(f"Features comportamentais: {comp_treino.shape[1]} colunas")

# --- Empilhar texto + comportamento numa matriz esparsa única -----------------
X_treino = sparse.hstack([X_texto_treino, sparse.csr_matrix(comp_treino.values)]).tocsr()
X_teste = sparse.hstack([X_texto_teste, sparse.csr_matrix(comp_teste.values)]).tocsr()
y_treino = df_treino["label"].values
y_teste = df_teste["label"].values

print(f"\nX_treino: {X_treino.shape} | X_teste: {X_teste.shape}")

# --- Validação: termos TF-IDF mais fortes em cada classe (só treino) ------------
media_spam = np.asarray(X_texto_treino[y_treino == 1].mean(axis=0)).ravel()
media_leg = np.asarray(X_texto_treino[y_treino == 0].mean(axis=0)).ravel()
termos = np.array(vectorizador.get_feature_names_out())
ordem_spam = media_spam.argsort()[::-1][:10]
ordem_leg = media_leg.argsort()[::-1][:10]
print("\nTermos mais fortes no SPAM (TF-IDF médio, treino):")
print(", ".join(termos[ordem_spam]))
print("\nTermos mais fortes no LEGÍTIMO (TF-IDF médio, treino):")
print(", ".join(termos[ordem_leg]))

Vocabulário TF-IDF: 5000 termos (unigramas + bigramas)
Features comportamentais: 23 colunas

X_treino: (80000, 5023) | X_teste: (20000, 5023)

Termos mais fortes no SPAM (TF-IDF médio, treino):
ref, verify, id, confirm, security, offer, limited, parcel, account, receive

Termos mais fortes no LEGÍTIMO (TF-IDF médio, treino):
thanks, please, review, notes, hi, attached, hello, weekly, please review, lunch tomorrow


In [0]:
# ==============================================================================
# STEP 6: TREINO DOS MODELOS — NAIVE BAYES E RANDOM FOREST
# ==============================================================================
# Objectivo: treinar os dois classificadores e compará-los de forma honesta
#            com validação cruzada SÓ no treino. O teste permanece intacto
#            para a avaliação final (STEP 7).
#
# Input:  X_treino, y_treino (80.000 emails × 5.023 features)
# Output: modelo_nb e modelo_rf treinados + previsões no teste (prev_nb, prev_rf)
#
# Operações:
#   - MultinomialNB (clássico para texto com TF-IDF) — CV 5 folds
#   - RandomForestClassifier (capta interações comportamentais) — CV 3 folds
#     (menos folds por custo computacional: 100 árvores × 5.023 features)
#   - Treino final de ambos no conjunto completo de treino
#   - Previsões guardadas para avaliação no STEP 7

import time
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# --- Definição dos modelos -----------------------------------------------------
# Naive Bayes Multinomial: referência para classificação de texto (rápido, forte
# em contagens/TF-IDF). Random Forest: captura interações (ex.: muitas URLs +
# autenticação fail) que o NB não vê.
modelo_nb = MultinomialNB()
modelo_rf = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=42
)

# --- Validação cruzada (apenas no treino) ----------------------------------------
print("Validação cruzada no TREINO (F1-score da classe spam):")
print("=" * 55)

t0 = time.time()
scores_nb = cross_val_score(modelo_nb, X_treino, y_treino, cv=5,
                            scoring="f1", n_jobs=-1)
print(f"Naive Bayes    : F1 = {scores_nb.mean():.4f} ± {scores_nb.std():.4f} "
      f"({time.time() - t0:.0f}s, 5 folds)")

t0 = time.time()
scores_rf = cross_val_score(modelo_rf, X_treino, y_treino, cv=3,
                            scoring="f1", n_jobs=-1)
print(f"Random Forest  : F1 = {scores_rf.mean():.4f} ± {scores_rf.std():.4f} "
      f"({time.time() - t0:.0f}s, 3 folds)")

# --- Treino final e previsões no teste -------------------------------------------
modelo_nb.fit(X_treino, y_treino)
modelo_rf.fit(X_treino, y_treino)

prev_nb = modelo_nb.predict(X_teste)
prev_rf = modelo_rf.predict(X_teste)

print("\nTreino final concluído — previsões no teste guardadas para o STEP 7.")

Validação cruzada no TREINO (F1-score da classe spam):
Naive Bayes    : F1 = 1.0000 ± 0.0000 (3s, 5 folds)
Random Forest  : F1 = 1.0000 ± 0.0000 (7s, 3 folds)

Treino final concluído — previsões no teste guardadas para o STEP 7.


In [0]:
# ==============================================================================
# P1: QUAL A PERCENTAGEM DE SPAM VS LEGÍTIMO?
# ==============================================================================
# Pergunta:    No dataset de 100.000 emails, qual a percentagem de spam
#              face a emails legítimos?
#
# Justificação: dimensionar o problema de negócio — a prevalência de spam
#               define o risco a que uma caixa de entrada está exposta e
#               contextualiza o custo de cada falso negativo do modelo.
#
# Resposta:    contagem e percentagem por classe (dados completos, ground
#              truth).

# Contagem e percentagem por classe (label: 1 = spam, 0 = legítimo)
contagens = df["label"].value_counts().sort_index()
percentagens = (contagens / len(df) * 100).round(2)

tabela_p1 = pd.DataFrame({
    "classe": ["Legítimo (label 0)", "Spam (label 1)"],
    "emails": contagens.values,
    "percentagem": percentagens.values,
})
display(tabela_p1)

# Conclusão da P1
print(f"RESPOSTA P1: {percentagens[1]}% dos emails são spam ({contagens[1]} de "
      f"{len(df)}) e {percentagens[0]}% são legítimos ({contagens[0]} de {len(df)}).")
print("O dataset está perfeitamente equilibrado (50/50) — 1 em cada 2 emails "
      "do corpus é spam.")

classe,emails,percentagem
Legítimo (label 0),50000,50.0
Spam (label 1),50000,50.0


RESPOSTA P1: 50.0% dos emails são spam (50000 de 100000) e 50.0% são legítimos (50000 de 100000).
O dataset está perfeitamente equilibrado (50/50) — 1 em cada 2 emails do corpus é spam.


In [0]:
# ==============================================================================
# P2: QUAIS OS TEMAS MAIS COMUNS NO SPAM VS NO LEGÍTIMO?
# ==============================================================================
# Pergunta:    Que temas dominam os emails de spam e os emails legítimos?
#
# Justificação: identificar os padrões temáticos do spam apoia regras de
#               filtragem, treino de equipas de segurança e conscientização
#               dos utilizadores (o que distingue um email suspeito).
#
# Resposta:    análise de tópicos por TF-IDF — os 15 termos com maior TF-IDF
#              médio em cada classe (dataset completo, ground truth).
#              Sem LDA: com emails de ~14 palavras e vocabulário temático
#              disjunto, o TF-IDF por classe identifica os tópicos de forma
#              mais direta e interpretável.

from sklearn.feature_extraction.text import TfidfVectorizer

# A limpeza (STEP 4) foi aplicada a df_treino/df_teste; para as perguntas de
# negócio aplicamos a mesma função ao dataset completo (ground truth, 100k)
if "texto_limpo" not in df.columns:
    df["texto_limpo"] = (
        df["subject"].fillna("") + " " + df["body_plain"].fillna("")
    ).apply(limpar_texto)

# TF-IDF sobre o texto limpo (assunto + corpo) — unigramas e bigramas
vec_p2 = TfidfVectorizer(max_features=5000, min_df=2, ngram_range=(1, 2))
X_p2 = vec_p2.fit_transform(df["texto_limpo"])
termos_p2 = np.array(vec_p2.get_feature_names_out())

# TF-IDF médio por classe (peso de cada tema em spam e em legítimo)
media_tf_spam = np.asarray(X_p2[df["label"].values == 1].mean(axis=0)).ravel()
media_tf_leg = np.asarray(X_p2[df["label"].values == 0].mean(axis=0)).ravel()

top_spam = media_tf_spam.argsort()[::-1][:15]
top_leg = media_tf_leg.argsort()[::-1][:15]

tabela_p2 = pd.DataFrame({
    "spam (tema/peso)": [f"{termos_p2[i]} ({media_tf_spam[i]:.3f})" for i in top_spam],
    "legítimo (tema/peso)": [f"{termos_p2[i]} ({media_tf_leg[i]:.3f})" for i in top_leg],
})
display(tabela_p2)

print("RESPOSTA P2: o spam gira em torno de urgência e recompensa — verify, ref,")
print("confirm, security, offer, limited, parcel, account — palavras de pressão e")
print("verificação de conta. O email legítimo gira em torno de trabalho de")
print("equipa — thanks, please, review, notes, attached, weekly — cortesia e")
print("rotina profissional. Os dois vocabulários praticamente não se cruzam.")

spam (tema/peso),legítimo (tema/peso)
ref (0.067),thanks (0.048)
verify (0.038),please (0.047)
id (0.037),review (0.045)
confirm (0.036),notes (0.040)
security (0.036),hi (0.040)
offer (0.035),attached (0.035)
limited (0.034),hello (0.034)
parcel (0.033),weekly (0.033)
account (0.032),please review (0.033)
receive (0.028),lunch (0.028)


RESPOSTA P2: o spam gira em torno de urgência e recompensa — verify, ref,
confirm, security, offer, limited, parcel, account — palavras de pressão e
verificação de conta. O email legítimo gira em torno de trabalho de
equipa — thanks, please, review, notes, attached, weekly — cortesia e
rotina profissional. Os dois vocabulários praticamente não se cruzam.


In [0]:
# ==============================================================================
# P3: QUAL O DOMÍNIO REMETENTE MAIS ASSOCIADO A SPAM?
# ==============================================================================
# Pergunta:    Que domínios de remetente estão mais associados a spam — em taxa
#              de spam e em volume absoluto de emails maliciosos enviados?
#
# Justificação: uma lista de bloqueio de domínios é a defesa mais barata de um
#               gateway de email; saber que domínios concentram o spam orienta
#               regras de filtragem e monitorização.
#
# Resposta:    análise por from_domain com dois cortes:
#              (a) domínios com volume relevante (≥100 emails) ordenados pela
#                  taxa de spam — evita falsos 100% em domínios com 2 emails;
#              (b) domínios com mais emails de spam em valor absoluto.

# Contagem de emails e de spam por domínio remetente
dominios = df.groupby("from_domain")["label"].agg(total="count", spam_enviados="sum")
dominios["taxa_spam_pct"] = (dominios["spam_enviados"] / dominios["total"] * 100).round(2)

print(f"Domínios distintos no dataset: {len(dominios)}")
print(f"Domínios com ≥100 emails: {(dominios['total'] >= 100).sum()}")

# (a) Domínios com volume relevante, ordenados pela taxa de spam
relevantes = dominios[dominios["total"] >= 100]
top_taxa = relevantes.sort_values(["taxa_spam_pct", "total"], ascending=False).head(10)
print("\n(a) TOP 10 DOMÍNIOS POR TAXA DE SPAM (≥100 emails enviados)")
display(top_taxa.reset_index())

# (b) Domínios que enviam mais spam em valor absoluto
top_absoluto = dominios.sort_values("spam_enviados", ascending=False).head(10)
print("\n(b) TOP 10 DOMÍNIOS POR VOLUME ABSOLUTO DE SPAM")
display(top_absoluto.reset_index())

# Conclusão da P3 (gerada a partir dos resultados)
mais_suspeito = top_taxa.iloc[0]
maior_volume = top_absoluto.iloc[0]
print(f"RESPOSTA P3: o domínio mais associado a spam em TAXA é '{top_taxa.index[0]}' "
      f"({mais_suspeito['taxa_spam_pct']:.1f}% de spam em {mais_suspeito['total']:.0f} "
      f"emails). Em VOLUME ABSOLUTO, '{top_absoluto.index[0]}' envia mais spam "
      f"({maior_volume['spam_enviados']:.0f} emails maliciosos, "
      f"{maior_volume['taxa_spam_pct']:.1f}% dos seus {maior_volume['total']:.0f} emails).")
print("Nota de leitura: taxas de 100% com volume alto indicam domínios "
      "exclusivamente maliciosos — candidatos diretos a lista de bloqueio.")

Domínios distintos no dataset: 18
Domínios com ≥100 emails: 18

(a) TOP 10 DOMÍNIOS POR TAXA DE SPAM (≥100 emails enviados)


from_domain,total,spam_enviados,taxa_spam_pct
icloud.com,7229,5812,80.4
gmail.com,7250,5770,79.59
yahoo.com,7206,5723,79.42
proton.me,7441,5896,79.24
outlook.com,7087,5612,79.19
accounts-mail.com,4892,1667,34.08
contoso.com,4984,1697,34.05
adatum.net,4960,1678,33.83
notifications.net,4967,1671,33.64
fabrikam.io,4884,1640,33.58



(b) TOP 10 DOMÍNIOS POR VOLUME ABSOLUTO DE SPAM


from_domain,total,spam_enviados,taxa_spam_pct
proton.me,7441,5896,79.24
icloud.com,7229,5812,80.4
gmail.com,7250,5770,79.59
yahoo.com,7206,5723,79.42
outlook.com,7087,5612,79.19
contoso.com,4984,1697,34.05
adatum.net,4960,1678,33.83
notifications.net,4967,1671,33.64
accounts-mail.com,4892,1667,34.08
fabrikam.io,4884,1640,33.58


RESPOSTA P3: o domínio mais associado a spam em TAXA é 'icloud.com' (80.4% de spam em 7229 emails). Em VOLUME ABSOLUTO, 'proton.me' envia mais spam (5896 emails maliciosos, 79.2% dos seus 7441 emails).
Nota de leitura: taxas de 100% com volume alto indicam domínios exclusivamente maliciosos — candidatos diretos a lista de bloqueio.


In [0]:
# ==============================================================================
# P4: OS EMAILS COM MAIS LINKS TENDEM A SER SPAM?
# ==============================================================================
# Pergunta:    O número de URLs no corpo do email está associado a spam?
#
# Justificação: links são o vetor principal de phishing (para onde a vítima
#               é levada a clicar). Se mais links implicar mais spam, o nº de
#               URLs é um sinal imediato e barato para regras de filtragem.
#
# Resposta:    taxa de spam por número de URLs + correlação entre num_urls e
#              a etiqueta (ponto-bisserial, equivalente a Pearson com uma
#              variável binária).

# Taxa de spam por número de URLs no corpo
links = df.groupby("num_urls")["label"].agg(total="count", taxa_spam="mean")
links["taxa_spam_pct"] = (links["taxa_spam"] * 100).round(2)
tabela_p4 = links.drop(columns="taxa_spam").reset_index()
tabela_p4 = tabela_p4.rename(columns={"num_urls": "urls no corpo"})
print("TAXA DE SPAM POR NÚMERO DE URLS NO CORPO")
display(tabela_p4)

# Correlação entre número de URLs e a etiqueta (1 = spam)
corr_links = df["num_urls"].corr(df["label"])
sem_links = links.loc[0, "taxa_spam_pct"] if 0 in links.index else None
com_links = df.loc[df["num_urls"] > 0, "label"].mean() * 100

print(f"\nCorrelação num_urls ↔ label: {corr_links:.3f}")
print(f"Taxa de spam sem links (num_urls=0): {sem_links}%" if sem_links is not None else "")
print(f"Taxa de spam com pelo menos 1 link: {com_links:.1f}%")
print(f"\nRESPOSTA P4: {'SIM' if com_links > sem_links else 'NÃO'} — emails com links "
      f"têm {com_links:.1f}% de spam contra {sem_links}% nos sem links; a "
      f"correlação é {corr_links:.3f} (positiva: mais links, mais spam).")
print("Nota de leitura: neste dataset a regra é determinística — TODOS os emails")
print("legítimos têm 0 URLs e TODOS os spam têm pelo menos 1. Em emails reais")
print("haveria legítimos com links (newsletters, notificações) e spam sem links,")
print("pelo que o número de URLs é um sinal forte, nunca uma regra absoluta.")

TAXA DE SPAM POR NÚMERO DE URLS NO CORPO


urls no corpo,total,taxa_spam_pct
0,50000,0.0
1,20030,100.0
2,29970,100.0



Correlação num_urls ↔ label: 0.918
Taxa de spam sem links (num_urls=0): 0.0%
Taxa de spam com pelo menos 1 link: 100.0%

RESPOSTA P4: SIM — emails com links têm 100.0% de spam contra 0.0% nos sem links; a correlação é 0.918 (positiva: mais links, mais spam).
Nota de leitura: neste dataset a regra é determinística — TODOS os emails
legítimos têm 0 URLs e TODOS os spam têm pelo menos 1. Em emails reais
haveria legítimos com links (newsletters, notificações) e spam sem links,
pelo que o número de URLs é um sinal forte, nunca uma regra absoluta.


In [0]:
# ==============================================================================
# P5: HÁ CORRELAÇÃO ENTRE O SCORE DO SPAMASSASSIN E A ETIQUETA?
# ==============================================================================
# Pergunta:    O score do SpamAssassin (x_spam_score) está correlacionado com
#              a classificação spam/legítimo do dataset?
#
# Justificação: o SpamAssassin é um filtro clássico de spam baseado em regras.
#               Saber até que ponto o seu score coincide com a etiqueta diz-nos
#               se o dataset foi construído a partir dele (correlação quase
#               perfeita) ou se a etiqueta é independente — o que muda a forma
#               como interpretamos o modelo treinado.
#
# Resposta:    correlação de Pearson entre x_spam_score e label, estatísticas
#              do score por classe e taxa de spam por intervalo de score.

# Estatísticas do score por classe
score_stats = df.groupby("label")["x_spam_score"].agg(
    media="mean", minimo="min", maximo="max"
).round(3)
tabela_score_classe = score_stats.reset_index()
tabela_score_classe["label"] = ["legítimo (0)", "spam (1)"]
tabela_score_classe = tabela_score_classe.rename(columns={"label": "classe"})
print("SCORE SPAMASSASSIN POR CLASSE")
display(tabela_score_classe)

# Taxa de spam por intervalo de score (bins de 1 ponto)
bins = pd.cut(df["x_spam_score"], bins=[-0.5, 0.5, 2, 4, 6, 8, 10, 100])
taxa_score = df.groupby(bins, observed=True)["label"].agg(total="count", taxa_spam="mean")
taxa_score["taxa_spam_pct"] = (taxa_score["taxa_spam"] * 100).round(2)
tabela_score_bins = taxa_score.drop(columns="taxa_spam").reset_index()
tabela_score_bins["x_spam_score"] = tabela_score_bins["x_spam_score"].astype(str)
print("\nTAXA DE SPAM POR INTERVALO DE SCORE")
display(tabela_score_bins)

# Correlação de Pearson entre score e etiqueta
corr_score = df["x_spam_score"].corr(df["label"])

# Verificação de determinismo: score 0 nos legítimos, >0 nos spam
leg_zero = (df.loc[df["label"] == 0, "x_spam_score"] == 0).mean() * 100
spam_pos = (df.loc[df["label"] == 1, "x_spam_score"] > 0).mean() * 100

print(f"\nCorrelação x_spam_score ↔ label: {corr_score:.3f}")
print(f"Legítimos com score 0: {leg_zero:.1f}%")
print(f"Spam com score > 0: {spam_pos:.1f}%")
print(f"\nRESPOSTA P5: SIM — correlação de {corr_score:.3f}. Os legítimos têm todos "
      f"score 0 ({leg_zero:.1f}%) e o spam tem sempre score > 0 ({spam_pos:.1f}%). "
      "A etiqueta foi muito provavelmente gerada a partir do próprio score — o "
      "que explica o F1 de 1,0 dos modelos e obriga a documentar esta "
      "dependência na publicação: em grande parte, o modelo herda a decisão do "
      "SpamAssassin em vez de aprender a detetar spam por si só.")

SCORE SPAMASSASSIN POR CLASSE


classe,media,minimo,maximo
legítimo (0),0.0,0.0,0.0
spam (1),8.0,4.0,12.0



TAXA DE SPAM POR INTERVALO DE SCORE


x_spam_score,total,taxa_spam_pct
"(-0.5, 0.5]",50000,0.0
"(2.0, 4.0]",31,100.0
"(4.0, 6.0]",12439,100.0
"(6.0, 8.0]",12557,100.0
"(8.0, 10.0]",12582,100.0
"(10.0, 100.0]",12391,100.0



Correlação x_spam_score ↔ label: 0.926
Legítimos com score 0: 100.0%
Spam com score > 0: 100.0%

RESPOSTA P5: SIM — correlação de 0.926. Os legítimos têm todos score 0 (100.0%) e o spam tem sempre score > 0 (100.0%). A etiqueta foi muito provavelmente gerada a partir do próprio score — o que explica o F1 de 1,0 dos modelos e obriga a documentar esta dependência na publicação: em grande parte, o modelo herda a decisão do SpamAssassin em vez de aprender a detetar spam por si só.


In [0]:
# ==============================================================================
# P6: OS EMAILS COM ANEXOS SÃO MAIS OU MENOS SUSPEITOS?
# ==============================================================================
# Pergunta:    A presença de anexos está associada a spam — os emails com
#              anexos são mais ou menos suspeitos do que os sem anexos?
#
# Justificação: anexos são vetor clássico de malware. Se os emails com anexos
#               fossem sistematicamente spam, uma regra de quarentena seria
#               justificada; se não, bloquear anexos penalizaria utilizadores
#               legítimos sem benefício de segurança.
#
# Resposta:    taxa de spam por presença de anexos, tipos de anexo mais comuns
#              por classe e comparação com a taxa base (50%).

# Taxa de spam por presença de anexos
anexos = df.groupby("has_attachments")["label"].agg(total="count", taxa_spam="mean")
anexos["taxa_spam_pct"] = (anexos["taxa_spam"] * 100).round(2)
anexos = anexos.drop(columns="taxa_spam")
anexos.index = ["sem anexos", "com anexos"]
tabela_anexos_resumo = anexos.reset_index().rename(
    columns={"index": "presença de anexos"}
)
print("TAXA DE SPAM POR PRESENÇA DE ANEXOS")
display(tabela_anexos_resumo)

# Tipos de anexo mais comuns em spam vs legítimos (só emails com anexos)
com_anexos = df[df["has_attachments"]]
# Extrai a extensão de cada ficheiro listado em attachment_types
def extensoes(tipos):
    if pd.isna(tipos):
        return []
    return [t.strip().rsplit(".", 1)[-1] for t in tipos.split(";")]

linhas = []
for classe, etiqueta in [("legítimo", 0), ("spam", 1)]:
    cont = pd.Series(
        [e for t in com_anexos.loc[com_anexos["label"] == etiqueta, "attachment_types"]
         for e in extensoes(t)]
    ).value_counts()
    for ext, n in cont.items():
        linhas.append({"classe": classe, "extensão": ext, "anexos": n})
tabela_anexos = (
    pd.DataFrame(linhas)
    .pivot_table(index="extensão", columns="classe", values="anexos",
                 fill_value=0)
    .astype(int)
    .sort_values("spam", ascending=False)
)
print("\nTIPOS DE ANEXO POR CLASSE (nº de ficheiros, só emails com anexos)")
display(tabela_anexos.reset_index())

# Conclusão da P6 (gerada a partir dos resultados)
taxa_com = anexos.loc["com anexos", "taxa_spam_pct"]
taxa_sem = anexos.loc["sem anexos", "taxa_spam_pct"]
mais_ou_menos = "MAIS suspeitos" if taxa_com > taxa_sem else "MENOS suspeitos"
print(f"\nRESPOSTA P6: emails COM anexos são {mais_ou_menos} — taxa de spam de "
      f"{taxa_com:.1f}% contra {taxa_sem:.1f}% nos sem anexos (taxa base do "
      "dataset: 50%).")
print("Nota de leitura: mesmo com taxa superior, existem milhares de emails "
      "legítimos com anexos — quarentena automática de todo o anexo não seria "
      "proporcional; o tipo de ficheiro deve pesar na decisão.")

TAXA DE SPAM POR PRESENÇA DE ANEXOS


presença de anexos,total,taxa_spam_pct
sem anexos,79858,46.86
com anexos,20142,62.44



TIPOS DE ANEXO POR CLASSE (nº de ficheiros, só emails com anexos)


extensão,legítimo,spam
docx,1798,1208
csv,1794,1206
pdf,1794,1205
scr,0,1204
docm,0,1196
pptx,1748,1191
lnk,0,1184
zip,0,1181
exe,0,1175
xlsx,1802,1157



RESPOSTA P6: emails COM anexos são MAIS suspeitos — taxa de spam de 62.4% contra 46.9% nos sem anexos (taxa base do dataset: 50%).
Nota de leitura: mesmo com taxa superior, existem milhares de emails legítimos com anexos — quarentena automática de todo o anexo não seria proporcional; o tipo de ficheiro deve pesar na decisão.


In [0]:
# ==============================================================================
# P7: QUAL A LÍNGUA COM MAIOR TAXA DE SPAM?
# ==============================================================================
# Pergunta:    Entre as 5 línguas do corpus (en, fr, it, es, de), qual regista
#              a maior taxa de spam?
#
# Justificação: se o spam se concentrasse numa língua, filtros e campanhas de
#               conscientização poderiam ser direcionados. Uma distribuição
#               uniforme indica campanhas globais, sem alvo linguístico.
#
# Resposta:    contagem e taxa de spam por língua (dataset completo), com
#              comparação contra a taxa base (50%) e amplitude entre línguas.

# Contagem e taxa de spam por língua — a língua é uma COLUNA visível da tabela
linguas = df.groupby("language")["label"].agg(total="count", taxa_spam="mean")
linguas["taxa_spam_pct"] = (linguas["taxa_spam"] * 100).round(2)
linguas["total_pct"] = (linguas["total"] / len(df) * 100).round(2)
nomes = {"en": "inglês", "fr": "francês", "it": "italiano", "es": "espanhol", "de": "alemão"}
tabela_p7 = (
    linguas.drop(columns="taxa_spam")
    .sort_values("taxa_spam_pct", ascending=False)
    .reset_index()
)
tabela_p7["language"] = [f"{nomes.get(l, l)} ({l})" for l in tabela_p7["language"]]
tabela_p7 = tabela_p7.rename(columns={"language": "língua"})
print("TAXA DE SPAM POR LÍNGUA (ordenada da maior para a menor taxa)")
display(tabela_p7)

# Conclusão da P7 (gerada a partir dos resultados)
top_lingua = tabela_p7.iloc[0]["língua"]
top_taxa = tabela_p7.iloc[0]["taxa_spam_pct"]
menor_taxa = tabela_p7.iloc[-1]["taxa_spam_pct"]
amplitude = top_taxa - menor_taxa
print(f"\nRESPOSTA P7: a língua com maior taxa de spam é o {top_lingua.split(' (')[0]} "
      f"({top_taxa:.2f}%). Contudo, a amplitude entre línguas é de apenas "
      f"{amplitude:.2f} pontos percentuais ({menor_taxa:.2f}%–{top_taxa:.2f}%), "
      "face a uma taxa base de 50%.")
print("Conclusão: o spam está distribuído de forma praticamente uniforme pelas "
      "5 línguas — campanhas globais sem alvo linguístico; filtrar por língua "
      "não traz vantagem de segurança.")

TAXA DE SPAM POR LÍNGUA (ordenada da maior para a menor taxa)


língua,total,taxa_spam_pct,total_pct
alemão (de),9934,51.02,9.93
francês (fr),10041,50.44,10.04
inglês (en),60032,49.93,60.03
espanhol (es),9981,49.69,9.98
italiano (it),10012,49.29,10.01



RESPOSTA P7: a língua com maior taxa de spam é o alemão (51.02%). Contudo, a amplitude entre línguas é de apenas 1.73 pontos percentuais (49.29%–51.02%), face a uma taxa base de 50%.
Conclusão: o spam está distribuído de forma praticamente uniforme pelas 5 línguas — campanhas globais sem alvo linguístico; filtrar por língua não traz vantagem de segurança.


In [0]:
# ==============================================================================
# STEP 7: AVALIAÇÃO FINAL NO TESTE — MÉTRICAS, CONFUSÃO E HONESTIDADE
# ==============================================================================
# Objectivo: avaliar os dois modelos no conjunto de teste (nunca visto),
#            comparar com as metas do projeto e investigar o resultado
#            perfeito da CV: quanto pesa o x_spam_score e o que vale o
#            pipeline NLP por si só (modelo só de texto).
#
# Input:  y_teste, prev_nb, prev_rf, modelo_rf, X_texto_treino, X_texto_teste
# Output: tabela de métricas vs metas, matrizes de confusão, top features RF
#
# Operações:
#   - Métricas no teste: F1, acurácia, recall, precisão (classe spam)
#   - Verificação contra as metas (F1 ≥ 0,90 | Acurácia ≥ 0,90 | Recall ≥ 0,85)
#   - Importância das features do Random Forest (top 15)
#   - Teste de honestidade: Naive Bayes treinado SÓ com TF-IDF (sem score)

from sklearn.metrics import (f1_score, accuracy_score, recall_score,
                             precision_score, confusion_matrix)

# --- Métricas finais no teste ----------------------------------------------------
METAS = {"F1 (spam) ≥ 0,90": lambda p: f1_score(y_teste, p) >= 0.90,
         "Acurácia ≥ 0,90": lambda p: accuracy_score(y_teste, p) >= 0.90,
         "Recall (spam) ≥ 0,85": lambda p: recall_score(y_teste, p) >= 0.85}


def avaliar(nome, previsoes):
    """Calcula as métricas da classe spam e verifica as metas do projeto."""
    m = {
        "F1 (spam)": f1_score(y_teste, previsoes),
        "Acurácia": accuracy_score(y_teste, previsoes),
        "Recall (spam)": recall_score(y_teste, previsoes),
        "Precisão (spam)": precision_score(y_teste, previsoes),
    }
    print(f"\n--- {nome} ---")
    for metrica, valor in m.items():
        print(f"  {metrica:18s}: {valor:.4f}")
    for meta, teste in METAS.items():
        print(f"  META {meta:22s} -> {'CUMPRIDA' if teste(previsoes) else 'NÃO CUMPRIDA'}")
    cm = confusion_matrix(y_teste, previsoes)  # linhas: real; colunas: previsto
    print("  Matriz de confusão (linhas=real, colunas=previsto):")
    print(f"                    previsto legítimo  previsto spam")
    print(f"    real legítimo      {cm[0, 0]:8d}        {cm[0, 1]:8d}")
    print(f"    real spam          {cm[1, 0]:8d}        {cm[1, 1]:8d}")


avaliar("NAIVE BAYES (texto + comportamentais)", prev_nb)
avaliar("RANDOM FOREST (texto + comportamentais)", prev_rf)

# --- Importância das features do Random Forest ----------------------------------
nomes_features = list(vectorizador.get_feature_names_out()) + list(comp_treino.columns)
importancias = modelo_rf.feature_importances_
top_idx = importancias.argsort()[::-1][:15]
tabela_imp = pd.DataFrame({
    "feature": [nomes_features[i] for i in top_idx],
    "importancia": importancias[top_idx].round(4),
}).reset_index(drop=True)
tabela_imp.index += 1  # ranking a partir de 1
print("\nTOP 15 FEATURES — RANDOM FOREST")
display(tabela_imp)
peso_score = importancias[nomes_features.index("x_spam_score")]
print(f"\nPeso do x_spam_score sozinho: {peso_score:.4f} "
      f"({peso_score * 100:.1f}% da importância total)")

# --- Teste de honestidade: modelo SÓ com texto (TF-IDF) --------------------------
# Se o texto sozinho também classificar bem, o pipeline NLP tem valor próprio;
# se não, ficamos a saber que o score SpamAssassin é que carrega o resultado.
modelo_nb_texto = MultinomialNB()
modelo_nb_texto.fit(X_texto_treino, y_treino)
prev_texto = modelo_nb_texto.predict(X_texto_teste)
avaliar("NAIVE BAYES — SÓ TEXTO (sem score/links/autenticação)", prev_texto)


--- NAIVE BAYES (texto + comportamentais) ---
  F1 (spam)         : 1.0000
  Acurácia          : 1.0000
  Recall (spam)     : 1.0000
  Precisão (spam)   : 1.0000
  META F1 (spam) ≥ 0,90       -> CUMPRIDA
  META Acurácia ≥ 0,90        -> CUMPRIDA
  META Recall (spam) ≥ 0,85   -> CUMPRIDA
  Matriz de confusão (linhas=real, colunas=previsto):
                    previsto legítimo  previsto spam
    real legítimo         10000               0
    real spam                 0           10000

--- RANDOM FOREST (texto + comportamentais) ---
  F1 (spam)         : 1.0000
  Acurácia          : 1.0000
  Recall (spam)     : 1.0000
  Precisão (spam)   : 1.0000
  META F1 (spam) ≥ 0,90       -> CUMPRIDA
  META Acurácia ≥ 0,90        -> CUMPRIDA
  META Recall (spam) ≥ 0,85   -> CUMPRIDA
  Matriz de confusão (linhas=real, colunas=previsto):
                    previsto legítimo  previsto spam
    real legítimo         10000               0
    real spam                 0           10000

TOP 15 FEATUR

feature,importancia
x_spam_score,0.1397
num_phone_numbers,0.0732
num_urls,0.0548
contains_tracking_token,0.0308
spf_result_pass,0.0276
ref,0.027
dkim_result_pass,0.027
dkim_result_fail,0.0163
thanks,0.013
weekly,0.0113



Peso do x_spam_score sozinho: 0.1397 (14.0% da importância total)

--- NAIVE BAYES — SÓ TEXTO (sem score/links/autenticação) ---
  F1 (spam)         : 1.0000
  Acurácia          : 1.0000
  Recall (spam)     : 1.0000
  Precisão (spam)   : 1.0000
  META F1 (spam) ≥ 0,90       -> CUMPRIDA
  META Acurácia ≥ 0,90        -> CUMPRIDA
  META Recall (spam) ≥ 0,85   -> CUMPRIDA
  Matriz de confusão (linhas=real, colunas=previsto):
                    previsto legítimo  previsto spam
    real legítimo         10000               0
    real spam                 0           10000


# Conclusão

## O que construí

Comecei com uma pergunta simples: consegue uma máquina distinguir um email de
phishing de um email normal?

Para responder, usei **100.000 emails reais de 2024**, em cinco línguas.
Construí tudo de raiz: descarreguei os dados, separei treino e teste antes de
qualquer transformação, limpei o texto com NLTK e transformei cada email em
números — as suas palavras (TF-IDF) e o seu comportamento (links, anexos,
remetente, autenticação). No fim, treinei dois modelos clássicos: **Naive
Bayes** e **Random Forest**.

## O resultado

Nos 20.000 emails de teste, que os modelos nunca tinham visto, os dois
acertaram em **todos**: F1 = 1,000, zero erros. As três metas definidas no
início (F1 ≥ 0,90 | Acurácia ≥ 0,90 | Recall ≥ 0,85) foram todas cumpridas.

## O que os dados me ensinaram

Percorri o corpus com sete perguntas. As respostas, em resumo:

| # | Pergunta | Resposta |
| --- | --- | --- |
| P1 | Quanto spam há? | 50% — 50.000 spam, 50.000 legítimos |
| P2 | Que temas dominam? | Spam: urgência e prémios (verify, offer, limited). Legítimo: trabalho de equipa (thanks, review, attached) |
| P3 | Que domínios enviam mais spam? | Em taxa: icloud.com (80,4%). Em volume: proton.me (5.896 emails) |
| P4 | Mais links, mais spam? | Sim — 100% dos emails com links são spam; 0% dos que não têm |
| P5 | O score SpamAssassin explica a etiqueta? | Sim — correlação 0,926; a etiqueta parece derivar do score |
| P6 | Anexos tornam o email suspeito? | Sim — 62,4% de spam com anexos contra 46,9% sem |
| P7 | Alguma língua tem mais spam? | Alemão (51%), mas a diferença entre línguas é de apenas 1,7 pontos |

## O resultado perfeito tinha um senão

Ao investigar, descobri que a etiqueta deste dataset foi muito provavelmente
criada a partir do score do SpamAssassin: todo o email legítimo tem score 0 e
todo o spam tem score acima de 0. Nenhum email legítimo tem links e os temas
das duas classes quase não se cruzam. Fiz o teste final — um Naive Bayes só
com o texto, sem mais nada. F1 = 1,000 na mesma.

Ou seja: em emails reais, ninguém atinge F1 de 1,0. Este notebook não esconde
esse limite — o valor está no pipeline, pronto a reaproveitar em dados reais.

## Para quem quiser continuar

O código está organizado em passos independentes (STEP 0 a 7, mais as sete
perguntas), cada um com objetivo, input e output declarados. Serve de ponto
de partida para dados reais: a estrutura mantém-se, só a expetativa muda.